# TunBERT + FastText — Combined Embedding Topic Model

**Method:** Concatenates **TunBERT** contextual embeddings with **FastText** subword embeddings trained directly on this corpus, compresses the combination with a small autoencoder, and feeds the result into a **Combined Topic Model (CombinedTM)**.

**Why this method:** TunBERT captures contextual/semantic meaning but can struggle with rare words, typos, and the heavy spelling variation typical of social media Tunisian dialect. FastText is trained on subword n-grams, which makes it robust to exactly that kind of noise (it can produce a reasonable vector even for a word it's never seen, from its component subwords). Combining the two aims to get contextual understanding *and* robustness to spelling variation.

**Pipeline:** preprocess → TunBERT embeddings → train FastText on this corpus → UMAP-reduce TunBERT to 128d → concatenate with FastText → autoencoder-compress to 64d → CombinedTM.

## 0. Setup

In [ ]:
!pip install numpy==1.26.4 gensim==4.3.3 --force-reinstall --upgrade
!pip install transformers sentence-transformers umap-learn hdbscan scikit-learn nltk pandas openpyxl contextualized-topic-models tensorflow spacy


## 1. Load the corpus

We start from the raw Tunisian dialect social media corpus. Each row is one post/message; we drop empty rows since they carry no signal for topic modeling.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel("../data/TunTap_Corpus.xlsx")

# Drop rows with missing text
df = df.dropna(subset=["message"])

# Reset index
df = df.reset_index(drop=True)

# Extract just the text column
message = df["message"].tolist()


## 2. Preprocessing

Tunisian dialect on social media mixes Arabic script, Arabizi (Latin transliteration with digits standing in for Arabic letters, e.g. `3` for `ع`), French loanwords, elongated words for emphasis (`hhhhh`, `mriiiigla`), and noise (URLs, mentions, hashtags, emojis). Standard NLP preprocessing pipelines aren't built for this, so we apply dialect-specific cleaning:

1. Remove custom Tunisian stopwords (functional words with no topical meaning)
2. Normalize character elongation (`mriiiigla` → `mrigla`)
3. Convert Arabizi digits back to their Arabic-letter equivalent (`3` → `a`, `7` → `h`, `5` → `kh`, `9` → `k`, `2` → `a`)
4. Strip URLs, mentions, hashtags, emojis and non-alphanumeric symbols
5. Normalize Arabic letter variants (e.g. `إأآا` → `ا`) so the same word isn't split across multiple spellings
6. Remove stopwords a second time (some appear only after cleaning) and drop any resulting empty lines

In [ ]:
# Read stopwords.txt into a set
with open("../data/tunisian_stopwords.txt", "r", encoding="utf-8") as f:
    all_stopwords = set(line.strip() for line in f if line.strip())


In [ ]:
def remove_custom_stopwords(text, stopwords_set):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stopwords_set]
    return " ".join(filtered)


In [ ]:
text = [remove_custom_stopwords(t, all_stopwords) for t in message]


In [ ]:
import re

def normalize_elongation(text):
    # Replace 2 or more repeated characters with 1
    return re.sub(r'(.)\1{1,}', r'\1', text)

def convert_tunisian_numbers(text):
    return (
        text.replace("3", "a")
            .replace("7", "h")
            .replace("5", "kh")
            .replace("9", "k")
            .replace("2", "a")
    )

def remove_numbers(text):
    # Remove all digits (0-9)
    return re.sub(r'\d+', '', text)

def clean_dual_script_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)

    # Remove emojis and symbols (non-alphanum + Arabic letters + space)
    text = re.sub(r"[^\u0621-\u063A\u0641-\u064A\w\s]", " ", text)

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # convert numbers like 3 → ع
    text = convert_tunisian_numbers(text)

    # Normalize elongation
    text = normalize_elongation(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    text = remove_numbers(text)

    return text


In [ ]:
cleaned_texts = [clean_dual_script_text(t) for t in text]

In [ ]:
final_texts = [remove_custom_stopwords(t, all_stopwords) for t in cleaned_texts]

In [ ]:
final_texts = [line for line in final_texts if line.strip() != '']


Check how many documents survived preprocessing:

In [ ]:
print(len(final_texts))


17287


## 4. TunBERT embeddings (contextual half)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model.eval()


In [ ]:
def embed_documents(docs, batch_size=32):
    embeddings = []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    with torch.no_grad():
        for i in range(0, len(docs), batch_size):
            batch = docs[i:i+batch_size]
            for doc in batch:
                inputs = tokenizer(doc, return_tensors="pt", truncation=True, padding=True).to(device)
                outputs = model.BertModel(**inputs, output_hidden_states=True)
                cls_embedding = outputs.last_hidden_state[:, 0, :]
                embeddings.append(cls_embedding.squeeze().cpu().numpy())
    return np.array(embeddings)


In [ ]:
print("Embedding documents with TunBERT...")
embeddings = embed_documents(final_texts)
print("Done embeddings. Shape:", embeddings.shape)


Embedding documents with TunBERT...
Done embeddings. Shape: (17287, 768)


## 5. Bag-of-words + coherence utilities

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import torch
import umap

vectorizer_model = CountVectorizer(
    stop_words=list(all_stopwords),
    tokenizer=lambda x: x.split(),
    ngram_range=(1, 2)
)

vectorizer_model.fit(final_texts)
X_bow = vectorizer_model.transform(final_texts)


## 3. Topic coherence utilities

We evaluate topic quality with two complementary metrics:
- **C_V coherence** — measures how semantically related the top words of a topic are, based on word co-occurrence in a sliding window (via `gensim`)
- **NPMI** (Normalized Pointwise Mutual Information) — a simpler, more interpretable co-occurrence measure computed directly on document sets, less sensitive to corpus size than raw PMI

Both are computed from `final_texts`, split into tokens.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from itertools import combinations
from collections import Counter
import math

tokenized_docs = [doc.split() for doc in final_texts]
dictionary = Dictionary(tokenized_docs)

def compute_cv_score(topics_list, tokenized_docs, dictionary):
    cm = CoherenceModel(topics=topics_list, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def compute_manual_npmi(topics_list, tokenized_docs):
    doc_sets = [set(d) for d in tokenized_docs]
    num_docs = len(doc_sets)
    word_doc_counts = Counter()
    for s in doc_sets:
        for w in s:
            word_doc_counts[w] += 1
    def topic_npmi(topic_words):
        vals = []
        for w1, w2 in combinations(topic_words, 2):
            p1 = word_doc_counts.get(w1, 0) / num_docs
            p2 = word_doc_counts.get(w2, 0) / num_docs
            co = sum(1 for s in doc_sets if w1 in s and w2 in s)
            p12 = co / num_docs
            if p12 > 0 and p1 > 0 and p2 > 0:
                pmi = math.log(p12 / (p1 * p2))
                npmi = pmi / (-math.log(p12))
                vals.append(npmi)
        return (sum(vals) / len(vals)) if vals else 0.0
    scores = [topic_npmi(topic) for topic in topics_list]
    return sum(scores) / len(scores), scores


## 6. Train FastText on this corpus, build the hybrid embedding

FastText is trained here (not pretrained) directly on the Tunisian dialect corpus with skip-gram, so its subword vocabulary matches the dialect rather than standard Arabic or English.

In [ ]:
import re
from gensim.models import FastText

def simple_tokenize(text):
    return re.findall(r'\b\w+\b', text)

tokenized_docs = [simple_tokenize(doc) for doc in final_texts]

fasttext_model = FastText(
    sentences=tokenized_docs,
    vector_size=200,
    window=10,
    min_count=5,
    sg=1,          # skip-gram
    epochs=50
)

def doc_embedding(doc):
    vectors = [fasttext_model.wv[w] for w in doc if w in fasttext_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(fasttext_model.vector_size)

fasttext_embeddings = np.array([doc_embedding(doc) for doc in tokenized_docs])

# Reduce TunBERT (768d) to 128d with UMAP, then concatenate with FastText (200d)
umap_model = umap.UMAP(n_components=128, random_state=42)
tunbert_reduced = umap_model.fit_transform(embeddings)

hybrid_embs = np.concatenate([tunbert_reduced, fasttext_embeddings], axis=1)


## 7. Compress the hybrid embedding with an autoencoder

The concatenated TunBERT + FastText vector is high-dimensional (128 + 200 = 328d). A small autoencoder compresses it to 64 dimensions, which keeps CombinedTM's contextual input compact while retaining the information the network finds useful for reconstruction.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_embs.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_embs, hybrid_embs, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(hybrid_embs)


## 8. Build the CTM dataset and train CombinedTM

`TopicModelDataPreparation` builds the vocabulary/BoW side; we feed our own compressed hybrid embeddings as the contextual side via `custom_embeddings`.

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM

tp = TopicModelDataPreparation("not-lain/TunBERT")

training_dataset = tp.fit(
    text_for_contextual=final_texts,
    text_for_bow=final_texts,
    custom_embeddings=compressed_embs
)

bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]

ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=contextual_size,
    n_components=50,
    num_epochs=5,
    batch_size=64,
    hidden_sizes=(128,),
    activation="relu",
    dropout=0.0,
    lr=2e-3)

ctm.fit(training_dataset)

topics_list = ctm.get_topic_lists(20)
cv_score = compute_cv_score(topics_list, tokenized_docs, dictionary)
avg_npmi, _ = compute_manual_npmi(topics_list, tokenized_docs)

print("\n📊 Evaluation Metrics: TunBERT + FastText")
print(f"CV (Topic Coherence): {cv_score:.3f}")
print(f"NPMI: {avg_npmi:.3f}")



📊 Evaluation Metrics: TunBERT + FastText
CV (Topic Coherence): 0.544
NPMI: 0.583


## Results

| Metric | Score |
|---|---|
| C_V Coherence | **0.544** |
| NPMI | **0.583** |

**Note:** the original exploration notebook this project is based on had a leftover duplicate training cell here that referenced undefined variables (`X_contextual`) — it was dead code left over from an earlier edit and has been removed in this cleaned-up version.